<p>which python<p>
<p>source .venv/bin/activate<p>
<p>jupyter notebook --no-browser --ip=0.0.0.0<p>

In [ ]:
!nvidia-smi

# STEP 1: Import Required Libraries

In [ ]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
from PIL import Image
import cv2
import os
from tqdm import tqdm
import warnings
import kagglehub
from scipy.spatial.distance import cdist
from scipy.stats import weibull_min
warnings.filterwarnings('ignore')

In [ ]:
# Check if CUDA is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")
torch.manual_seed(0)

# STEP 2: Load and Explore the Dataset

In [ ]:
path = "./competition/"
print("Path to dataset files:", path)

DATA_DIR = os.path.join(path, "image")
TRAIN_CSV = os.path.join(path, "train.csv")
SAMPLE_SUBMISSION = os.path.join(path, "sample_submission.csv")

LABELS = ['Ink scenery', 'comic', 'cyberpunk', 'futuristic UI', 'lowpoly', 'oil painting',
          'pixel', 'realistic', 'steampunk', 'water color']
LABEL2IDX = {label: idx for idx, label in enumerate(LABELS)}
IDX2LABEL = {idx: label for label, idx in LABEL2IDX.items()}

# STEP 4: Custom Dataset Class

In [ ]:
class ArtDataset(Dataset):
    def __init__(self, csv_path, img_dir, transform=None, is_test=False):
        self.data = pd.read_csv(csv_path)
        self.img_dir = img_dir
        self.transform = transform
        self.is_test = is_test
        self.has_labels = 'style' in self.data.columns and not is_test

        # Filter only known styles
        if self.has_labels:
            original_len = len(self.data)
            self.data = self.data[self.data['style'].isin(LABEL2IDX.keys())].reset_index(drop=True)
            filtered_len = len(self.data)
            print(f"Filtered out {original_len - filtered_len} unknown-labeled samples")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        uuid = self.data.iloc[idx]['uuid']
        img_path = os.path.join(self.img_dir, f"{uuid}.png")

        try:
            image = Image.open(img_path).convert("RGB")
        except Exception as e:
            print(f"Error loading image {img_path}: {e}")
            image = Image.new("RGB", (224, 224), (0, 0, 0))

        if self.transform:
            image = self.transform(image)

        if self.has_labels:
            style = self.data.iloc[idx]['style']
            label = LABEL2IDX[style]
            return image, label
        else:
            return image, uuid


# STEP 5: Data Transforms and Augmentation

In [ ]:
def pad_to_square(img, fill=0):
    w, h = img.size
    max_wh = max(w, h)
    pad_w = (max_wh - w) // 2
    pad_h = (max_wh - h) // 2
    padding = (pad_w, pad_h, max_wh - w - pad_w, max_wh - h - pad_h)
    return transforms.functional.pad(img, padding, fill=fill)

In [ ]:
# Define transforms for training (with augmentation)
train_transforms = transforms.Compose([
	transforms.Lambda(pad_to_square),
    transforms.Resize((224, 224)),
    transforms.RandomRotation(20),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Define transforms for validation/test (no augmentation)
val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("Data transforms defined:")
print("- Training: Pad, Resize, Rotation, Flip, ColorJitter, Normalize")
print("- Validation: Resize, Normalize only")

# STEP 6: Create Data Loaders

In [ ]:
# Load the training data and split it into train/validation
train_data = pd.read_csv(TRAIN_CSV)

# Filter only known styles
original_len = len(train_data)
train_data = train_data[train_data['style'].isin(LABEL2IDX.keys())].reset_index(drop=True)
filtered_len = len(train_data)
print(f"Filtered out {original_len - filtered_len} unknown-labeled samples from training data")

# Split the data into train and validation sets
train_df, val_df = train_test_split(
    train_data, 
    test_size=0.3,  # 30% for validation
    random_state=42, 
    stratify=train_data['style']  # Ensure balanced split across classes
)

print(f"Training samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")
print(f"Class distribution in training set:")
print(train_df['style'].value_counts())

# Save the split data to temporary CSV files
train_df.to_csv('temp_train.csv', index=False)
val_df.to_csv('temp_val.csv', index=False)

# Create datasets
train_dataset = ArtDataset('temp_train.csv', DATA_DIR, train_transforms)
val_dataset = ArtDataset('temp_val.csv', DATA_DIR, val_transforms)
test_dataset = ArtDataset(SAMPLE_SUBMISSION, DATA_DIR, val_transforms, is_test=True)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Train loader batches: {len(train_loader)}")
print(f"Validation loader batches: {len(val_loader)}")
print(f"Test loader batches: {len(test_loader)}")

# STEP 7: Visualize Sample Data

In [ ]:
def imshow(tensor, title=None):
    """Display a tensor as an image"""
    # Denormalize the image
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    
    image = tensor.clone().detach()
    image = image.numpy().transpose((1, 2, 0))
    image = std * image + mean
    image = np.clip(image, 0, 1)
    
    plt.imshow(image)
    if title:
        plt.title(title)
    plt.axis('off')

def visualize_batch(data_loader, num_samples=8):
    """Visualize a batch of images"""
    data_iter = iter(data_loader)
    images, labels = next(data_iter)
    
    plt.figure(figsize=(12, 6))
    for i in range(min(num_samples, len(images))):
        plt.subplot(2, 4, i + 1)
        imshow(images[i], title=f'Class {labels[i].item()}')
    plt.tight_layout()
    plt.show()

# Visualize training samples
print("Sample training images:")
visualize_batch(train_loader)

# STEP 8: Model Architecture

In [ ]:
class EfficientNetClassifier(nn.Module):
    def __init__(self, num_classes, pretrained=True):
        super(EfficientNetClassifier, self).__init__()
        
        self.backbone = models.efficientnet_b0(pretrained=pretrained)
        
        # Remove the original classifier
        self.backbone.classifier = nn.Identity()
        
        self.num_features = 1280  # EfficientNet-B0 final feature size
        
        # Custom classifier head
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(self.num_features, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, num_classes)
        )
    
    def forward(self, x):
        # Extract features (without original classifier)
        x = self.backbone.features(x)            # Output: (B, 1280, 7, 7)
        x = self.backbone.avgpool(x)             # Output: (B, 1280, 1, 1)
        x = torch.flatten(x, 1)                  # Output: (B, 1280)
        features = x                             # Save features for OpenMax
        logits = self.classifier(features)       # Output: (B, num_classes)
        return logits, features

num_classes = len(LABEL2IDX)

# Create model
model = EfficientNetClassifier(num_classes, pretrained=True)

# Move model to device
model = model.to(device)

# Display model architecture
print(f"Model created with {num_classes} classes")
print(f"Model moved to {device}")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# STEP 9: Loss Function and Optimizer

In [ ]:
# Define loss function
criterion = nn.CrossEntropyLoss()

# Define optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

# Learning rate scheduler
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

print("Training setup:")
print(f"Loss function: CrossEntropyLoss")
print(f"Optimizer: Adam (lr=0.001, weight_decay=1e-4)")
print(f"Scheduler: StepLR (step_size=10, gamma=0.1)")

# STEP 10: Training Functions

In [ ]:
def train_epoch(model, train_loader, criterion, optimizer, device):
    """
    Train the model for one epoch
    
    Returns:
        Average loss and accuracy for the epoch
    """
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    progress_bar = tqdm(train_loader, desc='Training')
    
    for batch_idx, (images, labels) in enumerate(progress_bar):
        images, labels = images.to(device), labels.to(device)
        
        # Zero gradients
        optimizer.zero_grad()
        logits, features = model(images)
        loss = criterion(logits, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Statistics
        running_loss += loss.item()
        _, predicted = torch.max(logits.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        # Update progress bar
        progress_bar.set_postfix({
            'Loss': f'{loss.item():.4f}',
            'Acc': f'{100.*correct/total:.2f}%'
        })
    
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100. * correct / total
    
    return epoch_loss, epoch_acc

def validate_epoch(model, val_loader, criterion, device):
    """
    Validate the model for one epoch
    
    Returns:
        Average loss, accuracy, and F1 score for the epoch
    """
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_predictions = []
    all_labels = []
    
    with torch.no_grad():
        progress_bar = tqdm(val_loader, desc='Validation')
        
        for images, labels in progress_bar:
            images, labels = images.to(device), labels.to(device)
            
            # Forward pass
            logits, features = model(images)

            loss = criterion(logits, labels)
            
            # Statistics
            running_loss += loss.item()
            _, predicted = torch.max(logits, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            # Store predictions for F1 score calculation
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
            # Update progress bar
            progress_bar.set_postfix({
                'Loss': f'{loss.item():.4f}',
                'Acc': f'{100.*correct/total:.2f}%'
            })
    
    epoch_loss = running_loss / len(val_loader)
    epoch_acc = 100. * correct / total
    
    # Calculate F1 score
    f1_macro = f1_score(all_labels, all_predictions, average='macro')
    f1_weighted = f1_score(all_labels, all_predictions, average='weighted')
    
    return epoch_loss, epoch_acc, f1_macro, f1_weighted

# STEP 11: Training Loop

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs=50):
    """
    Complete training loop with validation
    
    Returns:
        Training history dictionary
    """
    history = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': [],
        'val_f1_macro': [],
        'val_f1_weighted': []
    }
    
    best_f1 = 0.0
    best_model_state = None
    
    print(f"Starting training for {num_epochs} epochs...")
    print("-" * 50)
    
    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        
        # Training
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        
        # Validation
        val_loss, val_acc, val_f1_macro, val_f1_weighted = validate_epoch(model, val_loader, criterion, device)
        
        # Update learning rate
        scheduler.step()
        
        # Save best model
        if val_f1_weighted > best_f1:
            best_f1 = val_f1_weighted
            best_model_state = model.state_dict().copy()
            print(f"New best F1 score: {best_f1:.4f} F1_weighted, saving model state")
        
        # Store history
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_f1_macro'].append(val_f1_macro)
        history['val_f1_weighted'].append(val_f1_weighted)
        
        # Print epoch results
        print(f'Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%')
        print(f'Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%')
        print(f'Val F1 (Macro): {val_f1_macro:.4f}, Val F1 (Weighted): {val_f1_weighted:.4f}')
        print(f'Current LR: {optimizer.param_groups[0]["lr"]:.6f}')
        print("-" * 50)
        
        # Early stopping check
        if epoch > 10 and val_f1_macro < max(history['val_f1_macro'][-10:]) - 0.01:
            print("Early stopping triggered!")
            break
    
    # Load best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"Best model loaded with F1 score: {best_f1:.4f}")
    
    return history

In [ ]:
# Start training
history = train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs=20)

# STEP 12: Training Visualization

In [ ]:
def plot_training_history(history):
    """
    Plot training history
    """
    plt.figure(figsize=(15, 5))
    
    # Loss plot
    plt.subplot(1, 3, 1)
    plt.plot(history['train_loss'], label='Training Loss')
    plt.plot(history['val_loss'], label='Validation Loss')
    plt.title('Model Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    
    # Accuracy plot
    plt.subplot(1, 3, 2)
    plt.plot(history['train_acc'], label='Training Accuracy')
    plt.plot(history['val_acc'], label='Validation Accuracy')
    plt.title('Model Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    plt.grid(True)
    
    # F1 Score plot
    plt.subplot(1, 3, 3)
    plt.plot(history['val_f1_macro'], label='F1 Macro')
    plt.plot(history['val_f1_weighted'], label='F1 Weighted')
    plt.title('F1 Scores')
    plt.xlabel('Epoch')
    plt.ylabel('F1 Score')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.show()

# Plot training history
plot_training_history(history)

# STEP 13: Model Evaluation

In [ ]:
def evaluate_model(model, val_loader, device):
    """
    Comprehensive model evaluation
    """
    model.eval()
    all_predictions = []
    all_labels = []
    all_probabilities = []
    
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc='Evaluating'):
            images, labels = images.to(device), labels.to(device)
            
            model.eval()
            logits, features = model(images)
            probabilities = F.softmax(logits, dim=1)
            _, predicted = torch.max(logits, 1)
            
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probabilities.extend(probabilities.cpu().numpy())
    
    # Calculate metrics
    f1_macro = f1_score(all_labels, all_predictions, average='macro')
    f1_weighted = f1_score(all_labels, all_predictions, average='weighted')
    
    print(f"Final F1 Score (Macro): {f1_macro:.4f}")
    print(f"Final F1 Score (Weighted): {f1_weighted:.4f}")
    
    # Classification report
    print("\nClassification Report:")
    print(classification_report(all_labels, all_predictions))
    
    # Confusion matrix
    cm = confusion_matrix(all_labels, all_predictions)
    plt.figure(figsize=(12, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title('Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()
    
    return f1_macro, f1_weighted, all_probabilities

In [ ]:
# Evaluate the model
f1_macro, f1_weighted, val_probabilities = evaluate_model(model, val_loader, device)

# STEP 15: Handle Unknown Class (11th Class)

In [ ]:
class OpenMax:
    def __init__(self, num_classes, feature_dim, tailsize=20, distance_metric='euclidean'):
        self.num_classes = num_classes
        self.feature_dim = feature_dim
        self.tailsize = tailsize
        self.distance_metric = distance_metric
        
        # Store class means and weibull parameters
        self.class_means = {}
        self.weibull_params = {}
        
    def fit(self, features, labels):
        """
        Fit OpenMax parameters using training features and labels
        
        Args:
            features: numpy array of shape (n_samples, feature_dim)
            labels: numpy array of shape (n_samples,)
        """
        print("Fitting OpenMax parameters...")
        
        # Calculate class means
        for class_id in range(self.num_classes):
            class_mask = labels == class_id
            if np.sum(class_mask) > 0:
                self.class_means[class_id] = np.mean(features[class_mask], axis=0)
        
        # Calculate distances and fit Weibull distributions
        for class_id in range(self.num_classes):
            if class_id not in self.class_means:
                continue
                
            class_mask = labels == class_id
            class_features = features[class_mask]
            
            if len(class_features) == 0:
                continue
            
            # Calculate distances to class mean
            distances = cdist(class_features, [self.class_means[class_id]], 
                            metric=self.distance_metric).flatten()
            
            # Use only the largest distances (tailsize)
            if len(distances) >= self.tailsize:
                tail_distances = np.sort(distances)[-self.tailsize:]
            else:
                tail_distances = distances
            
            # Fit Weibull distribution to tail distances
            if len(tail_distances) > 1:
                # Weibull fitting
                shape, loc, scale = weibull_min.fit(tail_distances, floc=0)
                self.weibull_params[class_id] = (shape, loc, scale)
            else:
                # Fallback parameters if not enough samples
                self.weibull_params[class_id] = (2.0, 0.0, np.mean(tail_distances))
        
        print(f"OpenMax fitted for {len(self.weibull_params)} classes")
    
    def predict_openmax(self, features, logits, alpha=3, threshold=0.6):
        """
        Predict using OpenMax
        
        Args:
            features: numpy array of shape (n_samples, feature_dim)
            logits: numpy array of shape (n_samples, num_classes)
            alpha: number of top classes to revise
        
        Returns:
            openmax_probs: probabilities including unknown class
            predictions: predicted classes (num_classes = unknown)
        """
        n_samples = features.shape[0]
        openmax_scores = np.zeros((n_samples, self.num_classes + 1))  # +1 for unknown
        
        for i in range(n_samples):
            feature = features[i]
            logit = logits[i]
            logits = (logits - np.mean(logits)) / np.std(logits)
            # Calculate distances to all class means
            distances = {}
            for class_id in self.class_means:
                dist = cdist([feature], [self.class_means[class_id]], 
                           metric=self.distance_metric)[0, 0]
                distances[class_id] = dist
            
            # Get top alpha classes
            sorted_indices = np.argsort(logit)[::-1][:alpha]
            
            # Calculate weibull probabilities and revise scores
            revised_logits = logit.copy()
            unknown_score = 0.0
            
            for class_id in sorted_indices:
                if class_id in self.weibull_params and class_id in distances:
                    shape, loc, scale = self.weibull_params[class_id]
                    distance = distances[class_id]
                    
                    # Calculate Weibull probability
                    weibull_prob = weibull_min.cdf(distance, shape, loc=loc, scale=scale)
                    
                    # Revise the logit
                    revised_logits[class_id] = logit[class_id] * (1 - weibull_prob)
                    unknown_score += logit[class_id] * weibull_prob
            
            # Set OpenMax scores
            openmax_scores[i, :self.num_classes] = revised_logits
            openmax_scores[i, self.num_classes] = unknown_score  # Unknown class
        
        # Convert to probabilities
        openmax_probs = np.exp(openmax_scores) / np.sum(np.exp(openmax_scores), axis=1, keepdims=True)
        
        # Get predictions
        predictions = np.argmax(openmax_probs, axis=1)
        confidences = np.max(openmax_probs, axis=1)
        predictions[confidences < threshold] = self.num_classes
        
        return openmax_probs, predictions

In [ ]:
def extract_features_and_logits(model, data_loader, device):
    model.eval()
    feats, logs, labels = [], [], []

    with torch.no_grad():
        for images, second in tqdm(data_loader, desc="Extracting"):
            images = images.to(device)
            logits, features = model(images)

            feats.append(features.cpu().numpy())
            logs.append(logits.cpu().numpy())

            if isinstance(second, torch.Tensor):
                labels.append(second.cpu().numpy())

    feats = np.concatenate(feats)
    logs = np.concatenate(logs)

    return (feats, logs, np.concatenate(labels)) if labels else (feats, logs)

In [ ]:
# Initialize and fit OpenMax
print("Extracting features from validation data...")
val_features, val_logits, val_labels = extract_features_and_logits(model, val_loader, device)

In [ ]:
# Initialize OpenMax
feature_dim = val_features.shape[1]
openmax = OpenMax(num_classes=num_classes, feature_dim=feature_dim, tailsize=20)

# Fit OpenMax on validation data
openmax.fit(val_features, val_labels)

In [ ]:
# Extract features and logits from test data
print("Extracting features from test data...")
test_features, test_logits = extract_features_and_logits(model, test_loader, device)

# Make OpenMax predictions on test data
print("Making OpenMax predictions on test data...")
test_openmax_probs, test_openmax_preds = openmax.predict_openmax(test_features, test_logits, alpha=3)

print(f"Test predictions - Known: {np.sum(test_openmax_preds < num_classes)}")
print(f"Test predictions - Unknown: {np.sum(test_openmax_preds == num_classes)}")

final_predictions = test_openmax_preds

# STEP 16: Create Submission File

In [ ]:
def create_submission(test_dataset, predictions, output_file="submissionODIN.csv", unknown_label="UNK"):
    """
    Create submission file using test dataset and predictions
    
    Args:
        test_dataset: ArtDataset instance for test data
        predictions: Array of predicted class indices
        output_file: Output CSV filename
        unknown_label: Label for unknown/out-of-distribution samples
    """
    submission_data = []

    # Use the provided LABELS list for class mapping
    LABELS = ['Ink scenery', 'comic', 'cyberpunk', 'futuristic UI', 'lowpoly', 'oil painting',
              'pixel', 'realistic', 'steampunk', 'water color']
    
    # Create idx_to_class mapping from LABELS
    idx_to_class = {i: label for i, label in enumerate(LABELS)}

    for i in range(len(test_dataset)):
        # Get UUID from dataset
        uuid = test_dataset.data.iloc[i]['uuid']
        
        pred_class_id = predictions[i]

        # If prediction is an unknown class index, name it "unknown"
        if pred_class_id in idx_to_class:
            pred_class_name = idx_to_class[pred_class_id]
        else:
            pred_class_name = unknown_label

        submission_data.append({
            "uuid": uuid,
            "style": pred_class_name
        })

    submission_df = pd.DataFrame(submission_data)
    submission_df.to_csv(output_file, index=False)
    print(f"Saved submission to {output_file}")
    print(f"Submission shape: {submission_df.shape}")
    print(f"Sample predictions:")
    print(submission_df.head())

    return submission_df

In [ ]:
# Create submission file
submission = create_submission(test_dataset, final_predictions, output_file="submission_open.csv")

# STEP 17: Model Saving and Loading

In [ ]:
# Save the trained model
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'num_classes': num_classes,
    'f1_score': f1_weighted,
    'history': history
}, 'weigth_open.pth')

print("Model saved as 'weigth_open.pth'")

In [ ]:
# Updated load_model function to return history
def load_model_with_history(model_path, num_classes):
    """
    Load a saved model along with its training history
    """
    model = EfficientNetClassifier(num_classes)
    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    
    # Extract history if available
    history = checkpoint.get('history', None)
    f1_score = checkpoint.get('f1_score', 0.0)
    
    print(f"Model loaded from {model_path}")
    print(f"Best F1 score: {f1_score:.4f}")
    
    return model, history

In [ ]:
# Option to load model instead of training
USE_SAVED_MODEL = True

if USE_SAVED_MODEL:
    # Load the saved model
    loaded_model, history = load_model_with_history('weigth_open.pth', num_classes)
    model = loaded_model
    
    # If no history available, skip visualization or create dummy
    if history is None:
        print("No training history available - skipping training plots")
        # You can either skip plot_training_history() or create dummy data
        history = {
            'train_loss': [],
            'train_acc': [],
            'val_loss': [],
            'val_acc': [],
            'val_f1_macro': [],
            'val_f1_weighted': []
        }
else:
    # Use the freshly trained model (existing code continues)
    history = train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs=20)

In [ ]:
if USE_SAVED_MODEL:
	# Continue with evaluation
	f1_macro, f1_weighted, val_probabilities = evaluate_model(model, val_loader, device)

In [ ]:
if USE_SAVED_MODEL:
	# Initialize and fit OpenMax
	print("Extracting features from validation data...")
	val_features, val_logits, val_labels = extract_features_and_logits(model, val_loader, device)

	# Initialize OpenMax
	feature_dim = val_features.shape[1]
	openmax = OpenMax(num_classes=num_classes, feature_dim=feature_dim, tailsize=35)

	# Fit OpenMax on validation data
	openmax.fit(val_features, val_labels)
	
	# Extract features and logits from test data
	print("Extracting features from test data...")
	test_features, test_logits = extract_features_and_logits(model, test_loader, device)

	# Make OpenMax predictions on test data
	print("Making OpenMax predictions on test data...")
	test_openmax_probs, test_openmax_preds = openmax.predict_openmax(test_features, test_logits, alpha=3)

	print(f"Test predictions - Known: {np.sum(test_openmax_preds < num_classes)}")
	print(f"Test predictions - Unknown: {np.sum(test_openmax_preds == num_classes)}")

	final_predictions = test_openmax_preds
	submission = create_submission(test_dataset, final_predictions, output_file="submission_open.csv")

# STEP 19: Summary and Results

In [ ]:
print("\n" + "="*60)
print("TRAINING COMPLETE - SUMMARY")
print("="*60)
print(f"Final F1 Score (Macro): {f1_macro:.4f}")
print(f"Final F1 Score (Weighted): {f1_weighted:.4f}")
print(f"Number of test predictions: {len(final_predictions)}")
print(f"Model saved: weight.pth")
print(f"Submission file: submission_open.csv")
print(f"Device used: {device}")
print(f"Total training time: {len(history['train_loss'])} epochs")

print("\nModel Architecture:")
print(f"- Base model: EfficientNet-B0 (pre-trained)")
print(f"- Number of classes: {num_classes}")
print(f"- Total parameters: {total_params:,}")